# Fine-tuning Models on SageMaker

This notebook demonstrates how to fine-tune a pre-trained model on Amazon SageMaker to improve its performance on a specific task. Fine-tuning is a powerful technique that allows you to adapt a pre-trained model to your specific use case, often resulting in significant performance improvements.

## What is Fine-tuning?

Fine-tuning is the process of taking a model that has been pre-trained on a large dataset and then further training it on a smaller, task-specific dataset. This allows the model to adapt its learned features to the specific characteristics of your task.

Benefits of fine-tuning include:
- Improved accuracy on domain-specific tasks
- Faster training compared to training from scratch
- Better generalization with smaller datasets
- Adaptation to specific language patterns or terminology

## How Fine-tuning Complements Other Optimization Techniques

In previous notebooks, we explored various model optimization techniques:
- **Quantization**: Reducing the precision of model weights
- **Pruning**: Removing unnecessary connections in the model
- **Knowledge Distillation**: Creating smaller student models that learn from larger teacher models

Fine-tuning complements these techniques by:
1. Improving model quality before applying other optimizations
2. Recovering performance that might be lost during optimization
3. Adapting optimized models to specific domains

## What We'll Cover

In this notebook, we will:
1. Evaluate a pre-trained model on a sentiment analysis task
2. Prepare a dataset for fine-tuning
3. Configure and run a fine-tuning job on SageMaker
4. Deploy and evaluate the fine-tuned model
5. Compare performance before and after fine-tuning

Let's get started!

## Setup

First, let's install the specific packages needed for this fine-tuning notebook.

In [ ]:
# Install only the packages needed for this notebook
!pip install "torch==1.13.1" "transformers==4.26.0" "datasets==2.10.1" "boto3>=1.26.0" "sagemaker>=2.130.0" "pandas>=1.5.0" "numpy>=1.23.0" "matplotlib>=3.6.0" --quiet

Now, let's import the necessary libraries and set up our SageMaker environment.

In [ ]:
# Import only what we need for this notebook
import os
import json
import time
from datetime import datetime

# Data processing
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Machine learning
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    pipeline
)
from datasets import load_dataset

# AWS
import boto3
import sagemaker
from sagemaker.huggingface import HuggingFace
from sagemaker.huggingface.model import HuggingFaceModel

# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load workshop configuration if available
try:
    with open('workshop_config.json', 'r') as f:
        workshop_config = json.load(f)
    
    # Use configuration values
    base_model = workshop_config.get("base_model", "distilbert-base-uncased-finetuned-sst-2-english")
    task = workshop_config.get("task", "sequence-classification")
except FileNotFoundError:
    # Default values if config not found
    base_model = "distilbert-base-uncased-finetuned-sst-2-english"
    task = "sequence-classification"
    print("Workshop configuration not found. Using default values.")

In [ ]:
# Set up SageMaker session
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = boto3.session.Session().region_name
bucket = sagemaker_session.default_bucket()
prefix = "fine-tuning-workshop"

print(f"SageMaker session established in region: {region}")
print(f"Using S3 bucket: {bucket}")
print(f"Using S3 prefix: {prefix}")

## 1. Evaluate the Pre-trained Model

Before fine-tuning, let's evaluate the pre-trained model on our target task to establish a baseline.

In [ ]:
# Load the pre-trained model and tokenizer
print(f"Loading model: {base_model}")
tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForSequenceClassification.from_pretrained(base_model)

# Create a sentiment analysis pipeline
sentiment_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# Test with a few examples
test_texts = [
    "I love this product! It's amazing and works perfectly.",
    "This is terrible. Completely disappointed with the quality.",
    "It's okay, not great but not bad either.",
    "The customer service was excellent but the product was mediocre."
]

# Run inference
results = sentiment_pipeline(test_texts)

# Display results
for text, result in zip(test_texts, results):
    print(f"Text: {text}")
    print(f"Sentiment: {result['label']} (Score: {result['score']:.4f})\n")